In [1]:
# !pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.5 MB/s eta 0:00:00


In [2]:
# !git clone https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4
# !git switch gavriel-thesis

Cloning into 'topic_modeling_KBMI4'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 290 (delta 31), reused 53 (delta 19), pack-reused 220 (from 2)
Receiving objects: 100% (290/290), 57.64 MiB | 15.52 MiB/s, done.
Resolving deltas: 100% (121/121), done.
Filtering content: 100% (4/4), 294.40 MiB | 14.34 MiB/s, done.
/content/topic_modeling_KBMI4
Updating files: 100% (41/41), done.
Filtering content: 100% (9/9), 1.07 GiB | 12.90 MiB/s, done.
Branch 'gavriel-thesis' set up to track remote branch 'gavriel-thesis' from 'origin'.
Switched to a new branch 'gavriel-thesis'


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from umap import UMAP
from hdbscan import HDBSCAN

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cpu


In [ ]:
df = pd.read_csv("data/preprocessed_data.csv")

df.head()

,reviewId,bank,score,year,text
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,BCAMOBILE_REVIEWS,1,2025,ribet banget nih apk sumpah dikir verif dikit ...
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,BCAMOBILE_REVIEWS,2,2025,kenapa qris enggak bisa di pakai ya daritadi l...
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,BCAMOBILE_REVIEWS,1,2025,aplikasi nya sampah kenapa tiba tiba keluar te...
3,32d28ac6-c538-4749-969f-8af060915d96,BCAMOBILE_REVIEWS,3,2025,bagus
4,f99259b7-0d45-4422-b667-6c4fd521638b,BCAMOBILE_REVIEWS,1,2025,tidak ada solusi ketika ada kendala di persuli...


In [ ]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 193,827


# IndoBERT

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

e:\anaconda\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gavri\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downloa

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [ ]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [ ]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

100%|██████████| 6058/6058 [26:05<00:00,  3.87it/s]


In [ ]:
print(embeddings.shape)

(193827, 768)


In [ ]:
embeddings[0]

array([ 1.32068610e+00,  3.98074418e-01, -6.73392862e-02,  6.81021631e-01,
       -1.96820974e-01,  1.08589363e+00, -8.56553018e-01,  2.23273388e-03,
        8.79841328e-01,  3.65444899e-01, -3.26138228e-01,  4.72277194e-01,
       -8.05487573e-01,  1.67667508e-01, -2.56380171e-01, -4.40765619e-01,
       -4.77702200e-01,  2.61944145e-01, -2.86765069e-01, -2.72093844e-02,
        7.49712348e-01,  1.74879134e-02,  2.77186837e-02, -1.01489611e-01,
       -5.50755799e-01, -4.60477501e-01,  3.93649340e-01,  8.93991292e-01,
       -5.85758351e-02, -1.70096517e-01, -2.30777428e-01,  5.49784362e-01,
        2.69129515e-01,  5.88295817e-01, -7.31480896e-01,  9.41245556e-01,
        4.00836855e-01,  1.14819241e+00, -5.05113244e-01, -9.31886211e-02,
       -1.09638429e+00,  7.68689096e-01, -1.85003400e-01,  1.06485210e-01,
       -8.93403411e-01,  6.15286827e-01,  5.15412211e-01,  8.41604114e-01,
       -5.12369573e-01, -3.84789519e-02, -7.70254806e-02,  3.15342516e-01,
        1.43759996e-01,  

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.276495
Average Norm : 18.317768
Std Norm : 2.1703527


In [ ]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [ ]:
np.save(
    "indobert_embeddings.npy",
    embeddings
)

# BERTopic